In [3]:
import requests
import json

# Define the API endpoint
API_URL = "http://0.0.0.0:8000/query"

def query_agent(question: str):
    """
    Sends a question to the Agentic RAG API and returns the response.
    """
    payload = {
        "question": question
    }
    
    try:
        response = requests.post(API_URL, json=payload)
        response.raise_for_status()  # Check for HTTP errors
        
        # The API returns the result as a raw string according to main.py
        result = response.json()
        return result
        
    except requests.exceptions.RequestException as e:
        return f"Error connecting to API: {e}"



In [ ]:
import time
import json
import os
from datetime import datetime

# --- Configuration ---
QUESTIONS_PATH = "data/questions.json"
OUTPUT_DIR = "data"

def run_automated_evaluation():
    # 1. Load questions
    if not os.path.exists(QUESTIONS_PATH):
        print(f"Error: {QUESTIONS_PATH} not found.")
        return

    with open(QUESTIONS_PATH, "r") as f:
        questions_list = json.load(f)

    results = []
    print(f"Starting evaluation of {len(questions_list)} questions...")

    # 2. Iterate and Query
    for item in questions_list:
        q_id = item.get("id", "unknown")
        question_text = item.get("question", "")
        
        # Use metadata from questions.json if available, otherwise defaults
        user_type = item.get("persona_id", "persona_new_hire")
        scenario = item.get("scenario_id", "scenario_poor_relevance")
        
        print(f"Processing {q_id}...")
        
        start_time = time.time()
        
        try:
            # Calling the function already defined in your notebook
            answer = query_agent(question_text)
            success = True
            # Simple check if the function returned a connection error string instead of an answer
            if isinstance(answer, str) and answer.startswith("Error connecting to API"):
                success = False
        except Exception as e:
            answer = f"Exception occurred: {str(e)}"
            success = False
            
        end_time = time.time()
        duration_ms = int((end_time - start_time) * 1000)

        # 3. Format the response object according to your spec
        response_entry = {
            "question": question_text,
            "response": [
                answer,
                {
                    "num_chunks": 0,          # Placeholder as requested
                    "context_tokens": 0,      # Placeholder as requested
                    "completion_tokens": 0,   # Placeholder as requested
                    "embedding_tokens": 0     # Placeholder as requested
                }
            ],
            "success": success,
            "duration_ms": duration_ms,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "id": q_id,
            "user_type": user_type,
            "scenario": scenario,
            "judgment": "",                   # Empty as requested
            "reason": ""                      # Empty as requested
        }
        
        results.append(response_entry)

    # 4. Generate filename and save
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"responses_{timestamp_str}.json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    with open(output_path, "w") as f:
        json.dump(results, f, indent=4)

    print("-" * 30)
    print(f"SUCCESS: Evaluation complete.")
    print(f"Saved to: {output_path}")

# Run the automation
run_automated_evaluation()

In [5]:
import time
import json
import os
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# --- Configuration & Environment ---
# 1. Find the project root to load the .env file
# Assuming notebook is in 'synthetic-data-EDD/'
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent 
DOTENV_PATH = PROJECT_ROOT / ".env"

# 2. Load the environment variables
if DOTENV_PATH.exists():
    load_dotenv(dotenv_path=DOTENV_PATH)
else:
    print(f"⚠️ Warning: .env file not found at {DOTENV_PATH}")

# 3. Determine the Model Provider (defaults to 'groq' if not set or commented out)
MODEL_NAME = os.getenv("LLM_PROVIDER", "groq").lower()

QUESTIONS_PATH = "data/questions.json"
OUTPUT_DIR = "data"

def run_automated_evaluation_v2_dynamic():
    # 1. Load questions
    if not os.path.exists(QUESTIONS_PATH):
        print(f"Error: {QUESTIONS_PATH} not found.")
        return

    with open(QUESTIONS_PATH, "r") as f:
        questions_list = json.load(f)

    results = []
    print(f"Starting evaluation of {len(questions_list)} questions...")
    print(f"Detected Model Provider: {MODEL_NAME}")

    # 2. Iterate and Query
    for item in questions_list:
        q_id = item.get("id", "unknown")
        question_text = item.get("question", "")
        
        # Metadata from questions.json
        user_type = item.get("persona_id", "persona_new_hire")
        scenario = item.get("scenario_id", "scenario_poor_relevance")
        
        print(f"Processing {q_id}...")
        
        start_time = time.perf_counter()
        
        try:
            # Calling the function already defined in your notebook
            answer = query_agent(question_text)
            answer_content = answer if isinstance(answer, str) else str(answer)
        except Exception as e:
            answer_content = f"Exception occurred: {str(e)}"
            
        end_time = time.perf_counter()
        duration_ms = (end_time - start_time) * 1000

        # 3. Format according to the new schema
        response_entry = {
            "question": question_text,
            "response": [
                answer_content,
                "No source info available"
            ],
            "duration_ms": duration_ms,
            "timestamp": datetime.now().isoformat(),
            "model_provider": MODEL_NAME,
            "context_tokens": 0,
            "completion_tokens": 0,
            "id": q_id,
            "user_type": user_type,
            "scenario": scenario,
            "judgment": "", 
            "reason": "" 
        }
        
        results.append(response_entry)

    # 4. Generate filename and save
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"responses_{MODEL_NAME}_{timestamp_str}.json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    with open(output_path, "w") as f:
        json.dump(results, f, indent=4)

    print("-" * 30)
    print(f"SUCCESS: Evaluation complete for model '{MODEL_NAME}'.")
    print(f"Saved to: {output_path}")

# Execute the evaluation
run_automated_evaluation_v2_dynamic()

Starting evaluation of 4 questions...
Detected Model Provider: groq
Processing q_001...
Processing q_002...
Processing q_003...
Processing q_004...
------------------------------
SUCCESS: Evaluation complete for model 'groq'.
Saved to: data/responses_groq_20251224_203356.json
